In [1]:
import os
import clip
import torch

import numpy as np
from sklearn.linear_model import LogisticRegression
from torch.utils.data import DataLoader
from torchvision.datasets import CIFAR100
from tqdm import tqdm

In [2]:
# Load the model
device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load("ViT-B/32", device)

In [3]:
# Load the dataset
root = os.path.expanduser("~/.cache")
train = CIFAR100(root, download=True, train=True, transform=preprocess)
test = CIFAR100(root, download=True, train=False, transform=preprocess)

Files already downloaded and verified
Files already downloaded and verified


In [4]:
def get_features(dataset):
    all_features = []
    all_labels = []

    with torch.no_grad():
        for images, labels in tqdm(DataLoader(dataset, batch_size=100)):
            features = model.encode_image(images.to(device))

            all_features.append(features)
            all_labels.append(labels)

    return torch.cat(all_features).cpu().numpy(), torch.cat(all_labels).cpu().numpy()

In [5]:
# Calculate the image features
train_features, train_labels = get_features(train)
test_features, test_labels = get_features(test)

100%|██████████| 100/100 [00:20<00:00,  4.79it/s]


In [6]:
# Perform logistic regression
classifier = LogisticRegression(random_state=0, C=0.316, max_iter=1000, verbose=1)
classifier.fit(train_features, train_labels)

RUNNING THE L-BFGS-B CODE

           * * *

Machine precision = 2.220D-16
 N =        51300     M =           10

At X0         0 variables are exactly at the bounds

At iterate    0    f=  2.30259D+05    |proj g|=  7.66213D+02


 This problem is unconstrained.



At iterate   50    f=  3.38416D+04    |proj g|=  1.65101D+02

At iterate  100    f=  2.91905D+04    |proj g|=  3.47032D+02

At iterate  150    f=  2.83396D+04    |proj g|=  1.55932D+02

At iterate  200    f=  2.81616D+04    |proj g|=  7.13954D+01

At iterate  250    f=  2.81295D+04    |proj g|=  4.67183D+01

At iterate  300    f=  2.81220D+04    |proj g|=  1.13836D+01

At iterate  350    f=  2.81195D+04    |proj g|=  6.95854D+00

At iterate  400    f=  2.81173D+04    |proj g|=  1.48307D+01

At iterate  450    f=  2.81122D+04    |proj g|=  1.58092D+01

At iterate  500    f=  2.81013D+04    |proj g|=  3.91686D+01

At iterate  550    f=  2.80906D+04    |proj g|=  3.76695D+01

At iterate  600    f=  2.80859D+04    |proj g|=  3.97545D+00

At iterate  650    f=  2.80850D+04    |proj g|=  5.47459D+00

At iterate  700    f=  2.80847D+04    |proj g|=  1.49713D+00

At iterate  750    f=  2.80847D+04    |proj g|=  2.43619D+00

At iterate  800    f=  2.80845D+04    |proj g|=  8.72279D+00

At iter

/home/scliu/miniconda3/envs/RAG/lib/python3.8/site-packages/sklearn/linear_model/_logistic.py:460: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression(C=0.316, max_iter=1000, random_state=0, verbose=1)

In [7]:
# Evaluate using the logistic regression classifier
predictions = classifier.predict(test_features)
accuracy = np.mean((test_labels == predictions).astype(float)) * 100.0
print(f"Accuracy = {accuracy:.3f}")

Accuracy = 79.940
